In [3]:
from common import *
from canari_paper import *
Xf, Xc = cal_split(norm)
e1, e2 = CanariPaper(dim=8).fit(Xf)._eCF(Xf)
print(e1.mean(), comb(8 + 1, 1), e2.mean(), comb(8 + 4, 4))   # expect about 9 and 495

8.999990808427935 9 494.9993808151946 495


In [4]:
bestA, tableA = cv_paper(); print(bestA, {k: round(v, 3) for k, v in tableA.items()})

(6, 1.4142135623730951) {(6, 1.4142135623730951): np.float64(0.587), (6, 1.5): np.float64(0.586), (6, 2): np.float64(0.581), (8, 1.4142135623730951): np.float64(0.574), (8, 1.5): np.float64(0.573), (8, 2): np.float64(0.567), (10, 1.4142135623730951): np.float64(0.545), (10, 1.5): np.float64(0.544), (10, 2): np.float64(0.537)}


In [5]:
bestB = cv_tune(f_growth, [dict(dim=d) for d in (6, 8, 10)]); print(bestB)

(np.float64(0.7063392551037955), {'dim': 8}, (0.9, 0.999))


In [6]:
Xfit, Xcal = cal_split(norm); Xte = test[feats].values
PRED, SCORE = {}, {}
dim, k = bestA
mA = CanariPaper(dim=dim).fit(Xfit).calibrate(Xcal, k=k)
PRED["CANARI-paper"] = mA.predict(Xte); SCORE["CANARI-paper"] = mA.score(Xte)
print("tau_na", mA.tau_na, "| share flagged:", np.mean(PRED["CANARI-paper"] != "normal"))
_, params, q = bestB
fn = f_growth(Xfit, **params); t1, t2 = np.quantile(fn(Xcal), q)
SCORE["CANARI-growth"] = fn(Xte); PRED["CANARI-growth"] = zones(SCORE["CANARI-growth"], t1, t2)
print(pd.DataFrame([metrics(n, yte, PRED[n], SCORE[n]) for n in PRED]).set_index("model").round(3))
joblib.dump(dict(PRED=PRED, SCORE=SCORE, BEST=dict(A=bestA, B=bestB)), RES / "paper.pkl")

tau_na 271.49020134739953 | share flagged: 0.1175343080279766
               macroF1  near_P  near_R  near_F1    FAR  PR_AUC  ROC_AUC
model                                                                  
CANARI-paper     0.564    0.50   0.224    0.309  0.003   0.739    0.866
CANARI-growth    0.724    0.56   0.533    0.546  0.084   0.720    0.859


['C:\\Users\\Prisha\\Downloads\\ml_turbofan\\data preprocessing\\processed_output\\run_20261111_135602_491576\\results\\paper.pkl']